In [1]:
%load_ext autoreload
%autoreload 2

# Пример инференса Sequential Representation модели через FMLib

In [2]:
import os
import warnings

import hydra
import pyarrow.fs as fs
import torch
import torch.nn.attention as atten
from tqdm.autonotebook import tqdm

from fmlib.data.io import ParquetDataset
from fmlib.data.io.writer import PartitionedParquetWriter
from fmlib.pipeline import InferencePipeline
from fmlib.utils.loading import instantiate_model, load_config, load_model
from fmlib.utils.mode_context import mode_context

## Создание `ParquetDataset`
### Необходим для чтения файла по заданной структуре в метаданных. Позволяет загрузить датасет в оперативную память частями, а не полностью

In [3]:
model_folder = "configs/inference/sequence_representation_42M"
device = "cuda:0"

_, metadata = load_config(model_folder, config_file="parquet_metadata.yaml")

dataloader = ParquetDataset(
    source="/user/team/team_ai_avatar/rusakov/sbercampaign_pilots/january_pilot/pilot/pilot_slice_42m_chains",
    metadata=metadata,
    partition_size=1024,
    batch_size=16,
    filesystem=fs.HadoopFileSystem("hdfs://arnsdpsbx", port=0),
    device=device,
)

### Получение батча

In [4]:
batch = next(iter(dataloader))
batch.keys()

dict_keys(['epk_id_mask', 'epk_id', 'event_ids_mask', 'event_ids', 'evk_vnv_channel_group_mask', 'evk_vnv_channel_group', 'evk_vnv_evt_attr_1_mask', 'evk_vnv_evt_attr_1', 'evk_vnv_sale_product_class_id_mask', 'evk_vnv_sale_product_class_id', 'evk_vnv_sale_type_id_mask', 'evk_vnv_sale_type_id', 'evt_dttm_mask', 'evt_dttm', 'holidays_evt_attr_1_mask', 'holidays_evt_attr_1', 'pos_geo_evt_attr_1_mask', 'pos_geo_evt_attr_1', 'txn_evt_attr_1_mask', 'txn_evt_attr_1', 'txn_evt_attr_15_mask', 'txn_evt_attr_15', 'txn_evt_attr_2_mask', 'txn_evt_attr_2', 'txn_evt_attr_3_mask', 'txn_evt_attr_3', 'txn_evt_attr_4_mask', 'txn_evt_attr_4', 'txn_evt_attr_5_mask', 'txn_evt_attr_5', 'txn_evt_attr_9_mask', 'txn_evt_attr_9'])

## Инициализация Transform класса
### Этот класс необходим для обработки батча перед подачей в модель. Обратите внимание, что обработка может работать на GPU, это зависит от передаваемых в `Transform` тензоров
### Примеры операций, которые могут применяться в Transform:
- Фильтрация элементов
- Определение масок и маскирование
- Переименование/перегруппировка тензоров для совместимости с моделью
- Обработка вещественных значений - бакетизация, дискретизация
- Обработка категориальных значений - энкодинг

In [5]:
_, transform_config = load_config(model_folder, config_file="transform_config.yaml")
transform = hydra.utils.instantiate(transform_config.model)
test_transformed = transform(batch)
test_transformed.keys()

dict_keys(['events', 'epk_id', 'epk_id_mask'])

## Ручная компиляция модели

### Загрузка модели в формате PyTorch
#### Для успешной загрузки необходимо указать путь до папки. В этой папки должна лежать вся информация необходимая для загрузки модели - веса и конфиг

In [6]:
checkpoint_path = os.path.join(model_folder, "large_final.bin")
if os.path.exists(checkpoint_path):
    torch_model = load_model(model_folder)
else:
    warnings.warn("Fake checkpoint will be produced.", stacklevel=2)
    torch_model = instantiate_model(model_folder)

## Создадим объект класса `InferencePipeline`
#### Пайплайн будет отвечать за трансформацию данных и подачу этих данных в модель.
#### В результате работы пайплайна вы получите преобразованный батч после применения Transform'ов и результат работы модели.
#### Обратите внимание, что `InferencePipeline` может принимать несколько Transform'ов. В таком случае будет выведен результат работы для каждого Transform отдельно (массив результатов)
#### Класс `InferencePipeline` является наследником `torch.nn.Module`. Поэтому вы свободно можете применять `torch.compile` к нему

In [7]:
pipeline = InferencePipeline(model=torch_model, transform=transform).to(device)
pipeline = torch.compile(pipeline, dynamic=True)

In [8]:
with (
    atten.sdpa_kernel([atten.SDPBackend.EFFICIENT_ATTENTION]),
    torch.no_grad(),
    mode_context(pipeline, training=False),
    torch.amp.autocast("cuda", dtype=torch.bfloat16),
):
    transformed_batch, model_logits = pipeline(batch)

## Пример запуска полного цикла инференса и запись результата в `parquet`

In [9]:
import tempfile

directory = tempfile.TemporaryDirectory()

In [10]:
writer = PartitionedParquetWriter(base_path=f"{directory.name}/result", write_every_n_batch=32)

In [ ]:
with (
    atten.sdpa_kernel([atten.SDPBackend.EFFICIENT_ATTENTION]),
    torch.no_grad(),
    mode_context(pipeline, training=False),
    torch.amp.autocast("cuda", dtype=torch.bfloat16),
):
    for batch in tqdm(dataloader):
        transformed, model_logits = pipeline(batch)
        result = model_logits[0]
        result["epk_id"] = batch["epk_id"]
        result.pop("last_hidden_state")
        writer.write(result)

In [12]:
writer.close()
writer.base_path

'/tmp/tmp9y35pcnh/result'